# Workshop 2.3 (and Assignment)

In this workshop you will be putting all the pieces we learnt over the last week-and-a-half together - from basic Python, via data exploration and cleaning all the way up to modeling and evaluation.

We will start with a workshop, which will get you going.  This is followed by this week's assignment, which you are asked to work on in pairs, and submit by end of Sunday.

We will cover the following topics in today's workshop:

* Data cleaning revisited
* Pipelines
* Baseline model
* Mean Squared Error as figure of merit
* Cross validation
* Improving model fit through variable transformations and nonlinear terms

## Workshop 2.3

The first bit of code implements all that we have learnt about this data in the session on Monday;
in particular, the rounding of heights to whole feet, the excess of tall people, and the systematic
issues with the AVD measurements in the period 2003-2007.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
## If you're running this on Google Colab, use this to upload your file:

from google.colab import files
files.upload();  ## Upload 'Heart valve dissection data.csv'

In [ ]:
## Load your data.
## (If you're running Jupyter and the data is not in the directory where you started Jupyter, modify the path below)
df = pd.read_csv("Heart valve dissection data.csv")

## Drop the superfluous index columns
df = df.drop(['Unnamed: 0','Id'], axis=1)

## Remove rows with missing AVD values - we want to predict this, so imputation is not an option
df = df.loc[pd.notnull(df.AVD), :]

## set column type, otherwise it is interpreted as a string
df.loc[:,"dissection_date"] = pd.to_datetime(df["dissection_date"])

## remove people with imprecisely measured height
df = df[ (df.Height_cm != 91.44) & (df.Height_cm != 121.92) ]

## remove very tall people
df = df.loc[ df["Height_cm"] < 198, : ]

## remove people between the two dates that show an unexplained downward trend in residuals
df = df.loc[ (df["dissection_date"]>pd.Timestamp(2007,7,1)) | (df["dissection_date"] < pd.Timestamp(2003,1,1)), :]

Make a histogram plot of `Height_cm` to check that the issue of rounding towards
whole feet is now much improved.  Use `binwidth=1`.

Also make a two-dimensional histogram of `Height_cm` against `Age` to check that excess of tall people issue has
been resolved.  For a good visualisation, use parameters `binwidth=(1,3), cbar=True, norm=mpl.colors.LogNorm(), vmin=None, vmax=None, cmap="RdYlGn_r"`.



In [ ]:
# your code here

To slowly segue from data exploration into modeling (which philosophically is the same thing, really),
let's look at residuals for a very simple linear model for aortic valve diameter.

* Make a dataframe `data` with only the predictor
variables `Height_cm`, `Weight_kg` and `Age`, and a one-column dataframe `target` with just `AVD` in it.
* Run `fit(data, target)` on an instance of `sklearn.linear_model.LinearRegression` to fit a model for aortic valve diameter (AVD)
* Next, run `model.predict(data)` to get obtain the model's predictions on the training data.  

Note that we don't
use cross validation to avoid overtraining, for two reasons: this is only an exploratory analysis,
and this linear model has few parameters (4 in fact), limiting the possibilities of overtraining.

* Calculate the difference between `target` and the predictions, and assign this to a new column
called `residual` in the `df` dataframe.  

(You should of course be careful not ever to use `residual`
as explanatory variable in a model; but `df` will not be used as dataframe to model against, as it also
contains the target variable, so that's safe.  Beware not to add `residual` to the `data` dataframe -
that is asking for trouble later on!)

* Make a `histplot` of `residual` against `dissection_date` on the `x`-axis (both male and female together).
Use the parameters as in the `histplot` above, but use `bins=(100,25)`.


<div class="alert alert-info">

## Question 1
    
Do the residuals look roughly normally distributed, with no gross systematic trends over time?  (You will notice the gap where our exploratory data analysis has identified a systematic problem.)
    
</div>

* Calculate the mean squared error on the training data, using the function `sklearn.metrics.mean_squared_error` which takes a `target` and a `prediction`, both vectors, with matching length.

For the next question, it is useful to know that the probability that a (standard normally distributed, i.e. mean=0 and sd=1) `z`-score is more larger than `4.4` in absolute value, is about 1 in 100.000.


<div class="alert alert-info">

## Question 2
  
What is the standard deviation of the residuals?  Are there any predictions more than 4.4 standard deviations away from the observed value?  Would it be reasonable to class these as outliers?  


In [ ]:
# your code here

The model above is reasonable for a first look at the data, but we can do better.  For one,
we ignored the variable `Sex`, and we evaluated the model on training data.  Let's fix those two
issues, beginning with the model evaluation.

* Build a pipeline using `sklearn.pipeline.make_pipeline` composed of a `StandardScaler()` and `LinearRegression()`.

* Fit the model on all training data, and calculate the mean squared error again on the training data

<div class="alert alert-info">

## Question 3

Why is this exactly the same as we had before, despite the addition of the `StandardScaler()`?

</div>

* Using `cv=ShuffleSplit(n_splits = 100, train_size = 0.9)` as cross validation strategy, and `neg_mean_squared_error`
as scoring function, calculate the (negative) mean squared error again but now on 10% hold-out data.  

<div class="alert alert-info">

## Question 4

What is the mean of these scores across the 100 random splits?  Why is it slightly different each time?
Does it seem to be worse, better, or equal to the MSE on training data we calculated before?
Would you expect it to be slightly worse?  If you think it does not *seem* worse, why is that you think?

In [ ]:
# your code here

<div class="alert alert-block alert-warning">
    
One thing to note here is our use of the (negative) mean squared error (or equivalently, the [negative] root-mean-squared-error or standard deviation) as figure of merit.  When fitting classical linear model, the errors as **assumed** to follow a normal distribution with constant variance.  With this assumption, an unbiased estimator of the parameters of the model exists, and can be computed very efficiently - and this unbiased estimator turns out to minimize the mean squared error!  In other words, this figure of merit (or loss function) is intimately connected with the assumptions and algorithms underlying the linear model.  Therefore it makes sense to make this choice when using a linear model - but remember that you are implicitly making an assumption by doing so.  In this case the assumption seems quite well satisfied, with the exception of a handful of outliers.

</div>

Now we are adding the categorical variable `Sex`.

* Make a new dataframe `data`, now with `Height_cm`, `Weight_kg`, `Age` and `Sex`.

* Use a `ColumnTransformer` to make a preprocessor that applies a `OneHotEncoder` to categorical data, and `StandardScaler` to numerical data,
just as it was done for the logistic regression example in Workshop 2.2, and build a pipeline from this
preprocessor and a `LinearRegression`.  Call this pipeline `model2`.

* Run the same cross validation strategy as above on this model/pipeline.

<div class="alert alert-info">

## Question 5

How much does adding this one categorical variable improve the test data performance?

In [ ]:
# your code here

After this succes, let's try and improve the model further.  To see where we could make progress,
plot the residuals against each of the 3 numerical predictors.  If the model is optimal, no
trends should be visible; if you can spot trends, it means that the model can **not** see them, and
helping the model to see them -- by adding nonlinear terms, for instance -- would improve the fit.

* Calculate the residuals by fitting all training data to `model2`, calculating the difference between
`target` and predictions, and assign the results to a new column `residual`.

* For each of the three numerical variables (hint: use a for loop), make a 2D `histplot` of `residual` against
  the variable, with the now standard parameters, and
`bins=(50,20)`.

Add `plt.show()` after each plot, as otherwise you only see the last one.

<div class="alert alert-info">

## Question 6
    
Do you see any trends?

In [ ]:
# your code here

A lot of data is squished against the left axis -- a lot happens in the first year of life, much less so in
the later decades.  To better visualise (and also to better model what happens) it is useful to stretch
this area out.  

This can be done in several ways, one of which is using a square-root transformation.
The difference between `sqrt(1)` and `sqrt(0)` is 1, much larger than the difference between
`sqrt(11)` and `sqrt(10)` (about 0.15), effectively stretching the area around `0` while compressing
other regions.

Transform a variable by replacing `x=var` in the plot above by `x=np.sqrt(data[var])`.  Also, add a line `plt.xlabel("sqrt " + var)` or equivalent, to modify the x-axis label
to emphasize that the x-axis no longer shows the straight value, but the square root.

<div class="alert alert-info">

## Question 7
    
Do you see any trends?

In [ ]:
# your code here

The trends, if they were not visible before, are now quite clear, indicating that nonlinear terms are called
for.  It is also clear that a lot happens
around the value zero, and this "stretching" transformation is probably quite useful also for modeling.

Exploiting this observation to further improve the model is your assignment.


<div class="alert alert-success">

# Assignment

* Your task is to build the **best linear model** you can for predicting aortic valve diameter
* Assess your model with the cross validation pipeline outlined above
* Deadline for submission of (runnable, clearly annotated with running English text) .ipynb: **Sunday 23:59**
* You are asked to work in pairs.  Please include **both your names** in the filename of the notebook you submit.

    
Hint: improve your model by judiciously adding nonlinear terms.  The sections below are meant to help you on your way.  Using `ColumnTransformer`, `FunctionTransformer` and `PolynomialFeatures` you should be getting quite far.  Refer to the doc pages for detailed info on how to use them.  If you have time, feel free to go beyond this; google "scikit-learn building a better linear model" for inspiration.  The hints and questions below will get you going - but the assignment is to try harder, build the best model you can -- and to evaluate it properly and without bias.



</div>
    
Let us make a new model, but now including the stretching transformation, and (additional) nonlinear terms,
to the model.  To keep things simple (and to not over-complicate our model -- it is a good idea to proceed
step by step), let's do this for one variable only.  Make a new one-variable preprocessor as follows:

```python
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import FunctionTransformer

nonlinear_preprocessor = make_pipeline(
                                       FunctionTransformer(np.sqrt),
                                       PolynomialFeatures(degree=2, include_bias = False),
                                       StandardScaler())
```
This first applies the `sqrt` transformation, then adds powers of this variable (in this case, `sqrt(x)` and `sqrt(x)^2=x`), and finally normalizes each to mean 0 and variance 1.

* Introduce a variable `nonlinear_column` and assign the value `"Age"` to it.

* Use a `ColumnTransformer` to apply the preprocessor to `nonlinear_column`; the standard
  numerical preprocessor to the (remaining) numerical variables, and the categorical preprocessor to the
  (single) categorical variable.  Make sure that the nonlinear preprocessor is listed first, as
  otherwise the standard numerical preprocessor will gobble up all numerical columns.
  
* Build a pipeline using this preprocessor and `LinearRegression` as before, and run cross validation.
  
      
Does this improve the model?
    
Once this works, make a `for` loop over all three numerical variables, and print the mean squared error
  for all choices of numerical variable.

Which variable gives the best result?


For further improvements, consider using different nonlinear
transformations (a general power transformation?  `np.log1p`?), higher order terms, interactions.
Make sure you keep notes of what you did, including the unsuccessful experiments, and comment on them
in your submission. Make sure you assess all models in the same way, for a fair comparison.

In [ ]:
# your code here.  Please add explanatory text, aims, justifications, conclusions, ...